# StarCraft 2 Matchmaking Prototype

The goal is to test a new matchmaking system **without rounds** and **without divisions** (divisions may still be present in the frontend to report progress, but are not used for matchmaking itself).

The matchmaking must ensure that **all** bots

- play a "fair" number of matches;
- play predominantly matches against opponents of a similar skill;
- play matches against varied opponents within those of similar skill;
- play matches which are somewhat evenly distributed in time.

We need to verify that these requirements are fulfilled by any proposed matchmaking system by running it on a model of the real ladder and checking

- the match frequency distributions;
- the match-up rating-difference distributions;
- the match-up opponent distributions.

In case that the proposed matchmaking is non-greedy, i.e. has a mechanism to wait for bot to become available instead of starting a game immediately, we also need to check the ladder server utilization.

## Ladder Model

We need to simulate ratings updates and match durations in our ladder model.

### Ladder Model Ratings Updates

The expected score $E_\text{A}$ of a match between bots A and B (from A's perspective) with ratings $R_\text{A}$ and $R_\text{B}$ is given as

$$
E_\text{A} = \frac{1}{1 + 10^{\frac{R_\text{B} - R_\text{A}}{400}}}
$$

and update rule for bot A is

$$
\Delta R_\text{A} = K \cdot ( S_\text{A} - E_\text{A} ),
$$

where $K$ is the adjustment per game (set to 16 on AI Arena) and $S_\text{A}$ is A's actual score of the match (1 for a win, 0.5 for a draw and 0 for a loss).

The expected score $E_\text{A}$ is only a proxy for A's win probability if there is no possibility for a draw.
In the bot games of AI Arena, we frequently encounter draws, and we need to model them.

Additionally, ratings on AI Arena are often **not transitive**, i.e. A beating B almost surely and B beating C a.s. does not imply A beating C a.s. (which the ELO system assumes). For this reason, where we have data available we should use it to inform the outcome of a simulated match.
To get a smooth transition between no-data to data match-ups, we use the ELO system estimate (augmented with a global draw rate, $d$) as the prior and perform Bayesian inference with actual data, where available.

On AI Arena, matches have a **hard time limit of 60 minutes**. Games that reach this limit are always recorded as draws. We model this timeout event first, as it couples match duration and outcome. Concretely, for each simulated match we first determine whether it times out, and then:

- **If timeout:** duration is 60 minutes and the outcome is a draw.
- **If no timeout:** duration and outcome are sampled independently from the models below.

This means that both the outcome model and the duration model are fitted on **non-timeout games only**, and the draw rate $d$ refers to the non-timeout draw rate.

### Match Outcome Model

For the prior we use the 3-dimensional Dirichlet distribution (multivariate beta distribution) which is then updated with the observed number of non-timeout wins ($W$), losses ($L$) and draws ($D$), i.e.

$$
\begin{aligned}
&P_\text{A}^\text{ELO}(\text{win}) = (1 - d) \cdot E_\text{A},\\
&P_\text{A}^\text{ELO}(\text{loss}) = (1 - d) \cdot (1 - E_\text{A}),\\
&P_\text{A}^\text{ELO}(\text{draw}) = d;
\end{aligned}
$$

$$
\boldsymbol{\alpha}_\text{outcome} = \big(n_0 \cdot P_\text{A}^\text{ELO}(\text{win}) + W, \quad n_0 \cdot P_\text{A}^\text{ELO}(\text{draw}) + D, \quad n_0 \cdot P_\text{A}^\text{ELO}(\text{loss}) + L\big),
$$

$$
(p_\text{win},\, p_\text{draw},\, p_\text{loss}) \sim \text{Dir}(\boldsymbol{\alpha}_\text{outcome}),
$$

where $n_0$ determines the strength of the prior (in units of number of matches). In the following I will use $n_0 = 10$, which means that if we have real data for 10 matches, we consider the evidence from the real data as strong as the prior.

### Timeout Model

The timeout probability is modelled with a Beta-Binomial, using the global timeout rate $t$ as prior:

$$
p_\text{timeout} \sim \text{Beta}(n_0 \cdot t + T, \quad n_0 \cdot (1 - t) + N),
$$

where $T$ is the observed number of timeouts and $N$ the number of non-timeouts for the match-up.

### Match Duration Model

For non-timeout games, match durations are positive and right-skewed. We model them with a log-normal distribution truncated at 60 minutes:

$$
\log(\text{duration}) \sim \mathcal{N}(\mu, \sigma^2), \quad \text{duration} < 60.
$$

The truncation ensures that the duration model cannot produce games at the time limit, which are already handled by the timeout model. In practice, since the log-normal is fitted on non-timeout games (all shorter than 60 minutes), the truncation rarely rejects a sample.

The global parameters $\mu_0$ and $\sigma^2$ are fitted from the log-transformed durations of all non-timeout games. For a specific match-up with $n$ observed non-timeout durations with sample mean $\bar{x}$ (in log-space), the posterior mean is

$$
\mu_\text{duration} = \frac{n_0 \cdot \mu_0 + n \cdot \bar{x}}{n_0 + n},
$$

with posterior uncertainty $\sigma_\text{duration} = \sigma / \sqrt{n_0 + n}$. The variance $\sigma^2$ is kept fixed at the global estimate.

## Matchmaking System

The matchmaker is **greedy**: whenever a game slot becomes available, it scores all pairs of idle bots and starts the match with the highest score. The score for a candidate match between bots A and B is

$$
\text{score}(A, B) = w_\text{skill} \cdot f_\text{skill}(A, B) + w_\text{fair} \cdot f_\text{fair}(A, B) + w_\text{var} \cdot f_\text{var}(A, B) + w_\text{time} \cdot f_\text{time}(A, B),
$$

where the components correspond to the four matchmaking requirements:

### Skill Matching

Matches between similarly-rated bots should be preferred. We use a Gaussian decay over the rating difference:

$$
f_\text{skill}(A, B) = \exp\!\left(-\frac{(R_A - R_B)^2}{2\tau^2}\right),
$$

where $\tau$ controls how tolerant the system is of rating differences. Values in $[0, 1]$.

### Fairness

Bots that have played fewer games should be prioritised. Let $g_i$ be the number of games bot $i$ has completed in the last 24 hours and $\bar{g}$ the mean across all currently active bots:

$$
f_\text{fair}(A, B) = \max(\bar{g} - g_A,\; \bar{g} - g_B).
$$

Using the max preserves the sign of the deficit: it is positive when at least one bot is underplayed and negative when both are overplayed. The most underplayed bot drives the score, while the partner is chosen by the other components. Bots that recently joined the ladder naturally receive a temporary priority boost since their game count in the window is low.

### Opponent Variety

Repeated match-ups should be discouraged. Let $a_{AB}$ be the number of games A has played since last facing B, and $b_{AB}$ the same for B ($a_{AB} = b_{AB} = \infty$ if they have never met):

$$
f_\text{var}(A, B) = 1 - \exp\!\left(-\frac{\sqrt{a_{AB} \cdot b_{AB}}}{\lambda}\right),
$$

where $\lambda$ controls how quickly a past match-up is "forgotten". The geometric mean $\sqrt{a_{AB} \cdot b_{AB}}$ gives partial credit when one bot has diversified a lot but the other hasn't, while still requiring both to have moved on. Values in $[0, 1]$.

### Time Distribution

Bots that have been idle longer should be preferred. Let $t_i$ be the time since bot $i$ last finished a game. We use the RMS (root mean square) to aggregate the per-bot idle times:

$$
f_\text{time}(A, B) = \sqrt{\frac{t_A^2 + t_B^2}{2}}.
$$

The RMS is pulled toward the longer idle time, ensuring that no bot is left idle for too long, while still giving partial credit when both bots have been waiting. Idle times are always non-negative, so the sign issue that affects fairness does not apply here.

### Parameters

| Parameter | Role |
|-----------|------|
| $w_\text{skill},\, w_\text{fair},\, w_\text{var},\, w_\text{time}$ | Relative importance of each objective |
| $\tau$ | Rating-difference tolerance |
| $\lambda$ | Opponent-variety decay rate |

Since the components have different scales, the weights absorb normalisation — they are not directly comparable across components.